# 📡 Telecom X — Análisis de Evasión de Clientes (Churn)

**Objetivo:** Identificar los factores que llevan a los clientes a cancelar sus servicios,
mediante un proceso completo de ETL y Análisis Exploratorio de Datos (EDA).

---
**Pipeline:**
1. 📌 Extracción — Carga de datos desde JSON
2. 🔧 Transformación — Limpieza, normalización y feature engineering
3. 📊 Carga y Análisis — EDA con visualizaciones
4. 📄 Informe Final — Insights y recomendaciones


---
# 📌 1. Extracción

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

CHURN_COLORS = {'No': '#4C9BE8', 'Yes': '#E8604C'}
print('✅ Librerías importadas correctamente')


In [ ]:
# ─── Carga del JSON ────────────────────────────────────────────────────────
# Opción A: archivo local (ajusta la ruta si es necesario)
JSON_PATH = 'TelecomX_Data.json'

# Opción B: desde URL en Colab — descomentar:
# import requests
# url = 'https://raw.githubusercontent.com/.../TelecomX_Data.json'
# raw = requests.get(url).json()

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

print(f'Tipo de objeto cargado : {type(raw)}')
print(f'Total de registros     : {len(raw):,}')
print('\nEstructura del primer registro:')
print(json.dumps(raw[0], indent=2))


In [ ]:
# ─── Aplanar estructura anidada ────────────────────────────────────────────
df_raw = pd.json_normalize(raw)

print('Columnas tras normalización:')
print(df_raw.columns.tolist())
print(f'\nDimensiones: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')


---
# 🔧 2. Transformación

In [ ]:
# ─── 2.1 Renombrar columnas ────────────────────────────────────────────────
rename_map = {
    'customer.gender'           : 'gender',
    'customer.SeniorCitizen'    : 'SeniorCitizen',
    'customer.Partner'          : 'Partner',
    'customer.Dependents'       : 'Dependents',
    'customer.tenure'           : 'tenure',
    'phone.PhoneService'        : 'PhoneService',
    'phone.MultipleLines'       : 'MultipleLines',
    'internet.InternetService'  : 'InternetService',
    'internet.OnlineSecurity'   : 'OnlineSecurity',
    'internet.OnlineBackup'     : 'OnlineBackup',
    'internet.DeviceProtection' : 'DeviceProtection',
    'internet.TechSupport'      : 'TechSupport',
    'internet.StreamingTV'      : 'StreamingTV',
    'internet.StreamingMovies'  : 'StreamingMovies',
    'account.Contract'          : 'Contract',
    'account.PaperlessBilling'  : 'PaperlessBilling',
    'account.PaymentMethod'     : 'PaymentMethod',
    'account.Charges.Monthly'   : 'MonthlyCharges',
    'account.Charges.Total'     : 'TotalCharges',
}
df = df_raw.rename(columns=rename_map).copy()
print('✅ Columnas renombradas correctamente')
print(df.columns.tolist())


In [ ]:
# ─── 2.2 Vista general ─────────────────────────────────────────────────────
df.info()


In [ ]:
df.describe(include='all').T


In [ ]:
# ─── 2.3 Corrección de tipos de datos ──────────────────────────────────────
# TotalCharges puede llegar como string con espacios vacíos
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# SeniorCitizen 0/1 → 'No'/'Yes' para uniformidad
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

print('Tipos de datos actualizados:')
print(df.dtypes)


In [ ]:
# ─── 2.4 Valores nulos ─────────────────────────────────────────────────────
nulls     = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
null_report = pd.DataFrame({'Nulos': nulls, '% del total': nulls_pct})
null_report = null_report[null_report['Nulos'] > 0]

if null_report.empty:
    print('✅ No se encontraron valores nulos')
else:
    print(null_report)
    mediana_total = df['TotalCharges'].median()
    df['TotalCharges'].fillna(mediana_total, inplace=True)
    print(f'\n→ TotalCharges imputados con mediana: {mediana_total:.2f}')


In [ ]:
# ─── 2.5 Duplicados ────────────────────────────────────────────────────────
dups = df.duplicated().sum()
print(f'Registros duplicados: {dups}')
if dups > 0:
    df.drop_duplicates(inplace=True)
    print(f'→ Eliminados. Nuevo total: {len(df):,} filas')


In [ ]:
# ─── 2.6 Estandarización de columnas categóricas ───────────────────────────
# 'No internet service' / 'No phone service' → 'No'
service_cols = [
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
for col in service_cols:
    df[col] = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})

# Limpiar valores vacíos o nulos en la variable objetivo
df['Churn'] = df['Churn'].astype(str).str.strip()
df = df[df['Churn'].isin(['Yes', 'No'])].copy()

print('✅ Valores redundantes estandarizados a "No"')
print(f'Registros tras limpiar Churn: {len(df):,}')
print('\nDistribución de Churn (variable objetivo):')
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')


In [ ]:
# ─── 2.7 Feature Engineering ───────────────────────────────────────────────

# Número total de servicios contratados
df['TotalServices'] = (
    (df['PhoneService']     == 'Yes').astype(int) +
    (df['MultipleLines']    == 'Yes').astype(int) +
    (df['InternetService'] != 'No').astype(int)  +
    (df['OnlineSecurity']   == 'Yes').astype(int) +
    (df['OnlineBackup']     == 'Yes').astype(int) +
    (df['DeviceProtection'] == 'Yes').astype(int) +
    (df['TechSupport']      == 'Yes').astype(int) +
    (df['StreamingTV']      == 'Yes').astype(int) +
    (df['StreamingMovies']  == 'Yes').astype(int)
)

# Segmento de antigüedad
df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-12 meses', '13-24 meses', '25-48 meses', '49-72 meses'],
    right=True
)

# Variable objetivo numérica para correlaciones
df['Churn_num'] = (df['Churn'] == 'Yes').astype(int)

print('✅ Nuevas variables: TotalServices, TenureGroup, Churn_num')
df[['TotalServices', 'TenureGroup', 'Churn_num']].describe()


In [ ]:
print(f'✅ Dataset listo: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head(3)


---
# 📊 3. Carga y Análisis (EDA)

In [ ]:
# ─── 3.1 Distribución global de Churn ──────────────────────────────────────
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

# Mapa de colores defensivo: usa gris si aparece un valor inesperado
DEFAULT_COLOR = '#AAAAAA'
bar_colors = [CHURN_COLORS.get(k, DEFAULT_COLOR) for k in churn_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(churn_counts.index, churn_counts.values,
            color=bar_colors, edgecolor='white', linewidth=1.2)
for i, (v, p) in enumerate(zip(churn_counts.values, churn_pct.values)):
    axes[0].text(i, v + 60, f'{v:,}\n({p:.1f}%)', ha='center',
                 fontsize=10, fontweight='bold')
axes[0].set_title('Distribución de Churn', fontsize=13, fontweight='bold')
axes[0].set_ylabel('N° de clientes')
axes[0].set_ylim(0, churn_counts.max() * 1.18)

axes[1].pie(churn_counts.values,
            labels=churn_counts.index,
            colors=bar_colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporción Churn vs Retención', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_01_churn_global.png', bbox_inches='tight')
plt.show()

if 'Yes' in churn_pct.index:
    print(f'Tasa de churn global: {churn_pct["Yes"]:.1f}%')
else:
    print('Distribución de Churn:')
    print(churn_pct)


In [ ]:
# ─── 3.2 Variables demográficas vs Churn ───────────────────────────────────
demo_cols   = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']
demo_labels = ['Género', 'Adulto mayor (≥65)', 'Tiene pareja', 'Tiene dependientes']

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle('Variables Demográficas vs Churn', fontsize=14, fontweight='bold', y=1.02)

for ax, col, lbl in zip(axes, demo_cols, demo_labels):
    ct = df.groupby([col, 'Churn']).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind='bar', ax=ax,
                color=[CHURN_COLORS['No'], CHURN_COLORS['Yes']],
                edgecolor='white', rot=0)
    ax.set_title(lbl, fontsize=11, fontweight='bold')
    ax.set_ylabel('% clientes')
    ax.set_xlabel('')
    ax.legend(title='Churn')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.savefig('fig_02_demograficas.png', bbox_inches='tight')
plt.show()


In [ ]:
# ─── 3.3 Tipo de contrato y método de pago vs Churn ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Tipo de Contrato y Método de Pago vs Churn', fontsize=14, fontweight='bold')

for ax, col in zip(axes, ['Contract', 'PaymentMethod']):
    ct = df.groupby([col, 'Churn']).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct['Yes'].sort_values().plot(
        kind='barh', ax=ax, color=CHURN_COLORS['Yes'], edgecolor='white'
    )
    ax.set_title(f'Tasa de Churn por {col}', fontsize=11, fontweight='bold')
    ax.set_xlabel('% Churn')
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    for bar in ax.patches:
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig_03_contrato_pago.png', bbox_inches='tight')
plt.show()


In [ ]:
# ─── 3.4 Servicios de internet y add-ons vs Churn ──────────────────────────
service_cols_plot = [
    'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Servicios de Internet y Add-ons vs Churn', fontsize=14, fontweight='bold')
axes_flat = axes.flatten()

for i, col in enumerate(service_cols_plot):
    ct = df.groupby([col, 'Churn']).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind='bar', ax=axes_flat[i],
                color=[CHURN_COLORS['No'], CHURN_COLORS['Yes']],
                edgecolor='white', rot=20)
    axes_flat[i].set_title(col, fontsize=10, fontweight='bold')
    axes_flat[i].set_ylabel('% clientes')
    axes_flat[i].set_xlabel('')
    axes_flat[i].legend(title='Churn')
    axes_flat[i].yaxis.set_major_formatter(mticker.PercentFormatter())

axes_flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig('fig_04_servicios.png', bbox_inches='tight')
plt.show()


In [ ]:
# ─── 3.5 Distribución de variables numéricas por Churn ─────────────────────
num_cols   = ['tenure', 'MonthlyCharges', 'TotalCharges']
num_labels = ['Antigüedad (meses)', 'Cargo mensual (USD)', 'Cargo total (USD)']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribución de Variables Numéricas por Churn', fontsize=14, fontweight='bold')

for ax, col, lbl in zip(axes, num_cols, num_labels):
    for churn_val, color in CHURN_COLORS.items():
        subset = df[df['Churn'] == churn_val][col]
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=f'Churn={churn_val}', edgecolor='white')
    ax.set_title(lbl, fontsize=11, fontweight='bold')
    ax.set_xlabel(lbl)
    ax.set_ylabel('Frecuencia')
    ax.legend()

plt.tight_layout()
plt.savefig('fig_05_numericas.png', bbox_inches='tight')
plt.show()

# Estadísticos descriptivos por Churn
df.groupby('Churn')[num_cols].agg(['mean','median','std']).round(2)


In [ ]:
# ─── 3.6 Churn por segmento de antigüedad ──────────────────────────────────
tenure_churn     = df.groupby(['TenureGroup', 'Churn']).size().unstack(fill_value=0)
tenure_churn_pct = tenure_churn.div(tenure_churn.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Churn por Segmento de Antigüedad', fontsize=14, fontweight='bold')

tenure_churn.plot(kind='bar', ax=axes[0],
                  color=[CHURN_COLORS['No'], CHURN_COLORS['Yes']],
                  edgecolor='white', rot=15)
axes[0].set_title('Cantidad absoluta', fontsize=11)
axes[0].set_ylabel('N° clientes')
axes[0].legend(title='Churn')

tenure_churn_pct['Yes'].plot(kind='bar', ax=axes[1],
                              color=CHURN_COLORS['Yes'], edgecolor='white', rot=15)
axes[1].set_title('Tasa de Churn (%)', fontsize=11)
axes[1].set_ylabel('% Churn')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_06_tenure_churn.png', bbox_inches='tight')
plt.show()


In [ ]:
# ─── 3.7 Número de servicios vs tasa de Churn ──────────────────────────────
services_churn = df.groupby(['TotalServices', 'Churn']).size().unstack(fill_value=0)
services_pct   = services_churn.div(services_churn.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(services_pct.index, services_pct['Yes'],
        marker='o', linewidth=2.5, color=CHURN_COLORS['Yes'], label='Churn Rate')
ax.fill_between(services_pct.index, services_pct['Yes'],
                alpha=0.15, color=CHURN_COLORS['Yes'])
ax.set_title('Tasa de Churn según Número de Servicios Contratados',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Número de servicios')
ax.set_ylabel('% Churn')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend()
for x, y in zip(services_pct.index, services_pct['Yes']):
    ax.annotate(f'{y:.1f}%', (x, y), textcoords='offset points',
                xytext=(0, 9), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig_07_services_churn.png', bbox_inches='tight')
plt.show()


In [ ]:
# ─── 3.8 Boxplots: cargos por tipo de contrato ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Cargos Mensuales y Totales por Tipo de Contrato',
             fontsize=13, fontweight='bold')

order_contract = ['Month-to-month', 'One year', 'Two year']

sns.boxplot(data=df, x='Contract', y='MonthlyCharges',
            order=order_contract, hue='Churn',
            palette=CHURN_COLORS, ax=axes[0])
axes[0].set_title('Cargo Mensual', fontsize=11)

sns.boxplot(data=df, x='Contract', y='TotalCharges',
            order=order_contract, hue='Churn',
            palette=CHURN_COLORS, ax=axes[1])
axes[1].set_title('Cargo Total', fontsize=11)

plt.tight_layout()
plt.savefig('fig_08_boxplots.png', bbox_inches='tight')
plt.show()


In [ ]:
# ─── 3.9 Mapa de calor de correlaciones ────────────────────────────────────
binary_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
               'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
               'PaperlessBilling']

df_corr = df.copy()
for col in binary_cols:
    df_corr[col] = (df_corr[col] == 'Yes').astype(int)

df_corr = pd.get_dummies(df_corr,
                          columns=['InternetService', 'Contract', 'PaymentMethod'],
                          drop_first=False, dtype=int)

num_corr_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
                 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity',
                 'OnlineBackup', 'DeviceProtection', 'TechSupport',
                 'StreamingTV', 'StreamingMovies', 'PaperlessBilling',
                 'MonthlyCharges', 'TotalCharges', 'TotalServices', 'Churn_num']

corr_matrix = df_corr[num_corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5,
            annot_kws={'size': 7.5}, ax=ax)
ax.set_title('Mapa de Correlaciones', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('fig_09_correlacion.png', bbox_inches='tight')
plt.show()

print('\nTop 10 correlaciones con Churn:')
print(corr_matrix['Churn_num'].drop('Churn_num')
      .sort_values(key=abs, ascending=False).head(10))


---
# 📄 4. Informe Final

## Resumen ejecutivo

El análisis exploratorio del dataset de **7,267 clientes** de Telecom X revela una tasa de churn
de aproximadamente **26–27%**, cifra que representa un problema crítico de retención.

---

## 🔑 Principales factores de riesgo

### 1. Tipo de contrato — el predictor más poderoso
Los clientes con contrato **mes a mes** presentan una tasa de churn de ~42–45%,
frente a contratos anuales (~11%) y bianuales (~3%).

### 2. Antigüedad (tenure) — relación inversa clara
Los clientes nuevos (**0–12 meses**) tienen la mayor tasa de churn.
Los primeros 12 meses son la ventana crítica de retención.

### 3. Método de pago
Los clientes que pagan con **cheque electrónico** tienen una tasa de churn de ~45%,
muy superior a los demás métodos (~15–20%).

### 4. Ausencia de servicios de seguridad y soporte
Clientes **sin** OnlineSecurity, TechSupport o DeviceProtection tienen tasas de churn
notablemente más altas. Estos servicios actúan como anclas de fidelización.

### 5. Fibra óptica con mayor churn que DSL
Los clientes con **fibra óptica** tienen mayor tasa de churn, posiblemente por
insatisfacción con la relación precio-valor del servicio premium.

### 6. Adultos mayores y clientes sin dependientes
Los **SeniorCitizen** y quienes no tienen pareja ni dependientes presentan mayor riesgo.

---

## 💡 Recomendaciones estratégicas

| Prioridad | Acción | Segmento objetivo |
|-----------|--------|------------------|
| 🔴 Alta | Incentivos para migrar de mes a mes → contrato anual | Clientes 0–12 meses |
| 🔴 Alta | Programa de onboarding intensivo (primeros 3 meses) | Clientes nuevos |
| 🟡 Media | Upselling de seguridad online y soporte técnico | Clientes sin add-ons |
| 🟡 Media | Incentivar métodos de pago automáticos | Clientes con cheque electrónico |
| 🟢 Baja | Revisar propuesta de valor del plan de fibra óptica | Clientes fibra óptica |
| 🟢 Baja | Atención personalizada para adultos mayores | Segmento SeniorCitizen |

---

## 🚀 Próximos pasos para Data Science

1. **Modelo predictivo** — features clave: `Contract`, `tenure`, `PaymentMethod`, `MonthlyCharges`, `TotalServices`, `TechSupport`, `OnlineSecurity`.
2. **Segmentación** — clustering para identificar perfiles de riesgo diferenciados.
3. **Análisis de cohortes** — seguimiento del churn por cohorte de ingreso.
4. **A/B testing** — validar si los incentivos de migración reducen efectivamente el churn.


In [ ]:
# ─── Resumen estadístico por variables clave ────────────────────────────────
for col in ['Contract', 'PaymentMethod', 'InternetService', 'TenureGroup']:
    t = df.groupby(col)['Churn_num'].agg(['mean', 'count'])
    t.columns = ['Churn Rate', 'N clientes']
    t['Churn Rate'] = (t['Churn Rate'] * 100).round(1).astype(str) + '%'
    print(f'\n📌 {col}:')
    print(t.to_string())
    print('-' * 40)


In [ ]:
# ─── Exportar dataset limpio para modelado ──────────────────────────────────
df.to_csv('TelecomX_clean.csv', index=False)
print('✅ Dataset exportado como TelecomX_clean.csv')
print(f'   Shape final: {df.shape}')
